# Verify rung 0 yourself

This notebook re-derives every promoted claim of the rung-0 replicate ceiling from the
committed files in this repository — no cluster access, and no trust in any write-up.
Run all cells (Run → Run All Cells; about a minute on a laptop). Each section states a
claim in plain language, recomputes the number in front of you, and prints PASS or FAIL.
The same checks run in continuous integration (`tests/test_verify_rung0.py`), so the
branch stays green whether or not anyone opens this notebook — this notebook exists so
a reviewer can watch the re-derivation happen rather than take it on faith.

To run it: from the repository root, `uv sync --extra dev`, then
`uv run jupyter lab docs/tasks/rung0-replicate-ceiling/verify.ipynb`.
The terminal form of the same battery is `uv run python scripts/verify_rung0.py`.

**What is not checkable locally, stated rather than hidden:** the gene and drug panel
files live on the Alpine cluster and are pinned by checksum in the provenance record, so
the declared panel size (14,121 genes) is a recorded input property here, not a
recomputable one; and the 1,026 data shards sit on cluster scratch, so their integrity
reduces locally to the committed shard manifest hashing to the promoted record's data
version (checked below).

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
assert repo is not None, 'run this notebook from inside the repository (its own folder works)'
spec = importlib.util.spec_from_file_location('verify_rung0', repo / 'scripts' / 'verify_rung0.py')
assert spec is not None and spec.loader is not None
vr = importlib.util.module_from_spec(spec)
sys.modules['verify_rung0'] = vr
spec.loader.exec_module(vr)
print(f'repository: {repo}')

## 1. The promoted number cannot have been edited

The promoted table carries a provenance record with its checksum written at promotion
time. Recomputing the checksum from the file as it sits in the repository proves the
number under discussion is the number that was promoted — same for the cluster job log,
and the task-folder copy must be byte-identical to the promoted copy.

In [ ]:
print(vr.render(vr.check_promoted_hashes(repo)))

## 2. The input data are pinned

The 1,026 downloaded data shards are described by a committed manifest — each shard's
path, size, and checksum, one line per shard. The checksum of that manifest is the
content hash the promoted record pins as its data version, so the data the run consumed
is fixed by files you can hash yourself.

In [ ]:
print(vr.render(vr.check_tranche_content_hash(repo)))

## 3. The headline's arithmetic holds together

From the promoted table alone: the Spearman-Brown full-data ceiling must equal
2r/(1+r) of the split-half mean, the quartiles must bracket the median, both mismatch
floors must sit below the observed mean in the right order, the effect-size terciles
must rise monotonically (the in-run positive control), and both detection thresholds
must sit below the observed mean (the result is not a power artifact).

In [ ]:
print(vr.render(vr.check_headline_consistency(repo)))

## 4. The 1,600 scored conditions are exactly the splittable ones

The run measured the composition of the pool it consumed (rather than asserting it).
From that table: 1,650 candidate (cell line, drug) conditions; exactly 50 have all
their replicate plates in one half — all of them Ribociclib, which has a single plate
throughout the pool and cannot be split — and 1,650 − 50 equals the promoted pair count.

In [ ]:
print(vr.render(vr.check_pool_arithmetic(repo)))

## 5. The significance survives the dependence check

The reported p-values assume the mismatched-pair null draws behave like an exchangeable
pool although they reuse half-profiles. The task measured that assumption with shuffle
(derangement) checks — 500 permutations per comparison type, every per-permutation mean
committed. Recompute from those files: each null's mean and spread, the exact p, that
the true pairing beats all 500 shuffles in every stratum, that the any-pair design
effect (the dependence's measured distortion) is below one — the shortcut was cautious,
not generous — and that two entirely different sampling mechanisms land on the same
mismatch floors.

In [ ]:
print(vr.render(vr.check_derangement(repo)))

## 6. Reliability is broadly distributed, led by stress-response genes

The unpromoted per-gene diagnostic: 13,886 panel genes, each correlated across
conditions between the two plate halves. The write-up's numbers — 97.0% positive,
median 0.146, and a top of the table made of heat-shock and immediate-early
stress-response transcripts — recomputed from the committed table.

In [ ]:
print(vr.render(vr.check_per_gene_diagnostic(repo)))

## 7. The summary says what the artifacts say

Every number in `summary.md`'s evidence table and caveat paragraphs, parsed out of the
document and matched to the artifact it came from. A transcribed number that drifts
from its artifact fails here mechanically instead of waiting for someone to notice.

In [ ]:
print(vr.render(vr.check_summary_table(repo)))

## 8. The instruments themselves are validated

Everything above checks the numbers; this checks the code that produced them. The
known-answer suite plants answers in synthetic data and requires the real, shipped
functions to recover them: a synthetic replicate pool with reliability 0.8 planted must
come out 0.8 through the real measurement, a signal-free pool must come out null, the
power calculation must match the closed-form normal-theory answer, and the one
statistical defect this project has actually shipped (comparing an aggregate against
single draws) is pinned by a test that demonstrates the wrong form failing.

In [ ]:
result = subprocess.run(
    ['uv', 'run', 'pytest', '-m', 'known_answer', '-q'],
    cwd=repo, capture_output=True, text=True,
)
print(result.stdout[-2500:] + result.stderr[-500:])
assert result.returncode == 0, 'known-answer controls failed'

## Everything at once

The full battery, with the overall verdict.

In [ ]:
checks = vr.run_all_checks(repo)
print(vr.render(checks))
assert all(c.ok for c in checks), 'at least one promoted claim failed to recompute'
print('\nEvery promoted claim recomputed from the committed artifacts.')
print('You have re-derived rung 0, not read about it.')